# ImplementForge — FreshCart Under Pressure
## COSE 278: Implementing Systems — Day 4

**Run cells in order. Do not skip cells.**

Your instructor has pre-loaded the game engine from GitHub.
You do not need to modify any cell marked "do not modify".


## ⚙️ CELL 0 — Setup
*Run this first. Do not modify.*


In [ ]:
# Download game engine from GitHub — do not modify
GITHUB_BASE = "https://raw.githubusercontent.com/rickghome/278game/main"

import urllib.request, os, pickle 

for filename in ["if_engine.py", "if_cards.py"]:
    urllib.request.urlretrieve(f"{GITHUB_BASE}/{filename}", filename)
    print(f"  ✅  {filename} loaded")

exec(open("if_engine.py").read())
exec(open("if_cards.py").read())

print("\n✅  ImplementForge loaded. Scroll to CELL 0b.")


## 💾 CELL 0b — Save / Restore
*Run after each phase to protect against session timeout.*


In [ ]:
# Save/restore — do not modify

def save_game(game):
    with open('game_state.pkl', 'wb') as f:
        pickle.dump(game, f)
    print(f"💾  Saved — {game['team_name']}")
    print(f"    Income: ${sum(game['income'].values()):,}  |  Trust: {game['trust_score']}/100")

def restore_game():
    if not os.path.exists('game_state.pkl'):
        print("❌  No saved game found. Start from CELL 1.")
        return None
    with open('game_state.pkl', 'rb') as f:
        game = pickle.load(f)
    print(f"✅  Restored — {game['team_name']}")
    print(f"    Income: ${sum(game['income'].values()):,}  |  Trust: {game['trust_score']}/100")
    print(f"    Active seeds: {game['seeds']}")
    return game

# To save:     save_game(game)
# To restore:  game = restore_game()
print("💾  Save/restore ready. Run save_game(game) after each phase.")


## 🏷️ CELL 1 — Team Setup


In [ ]:
TEAM_NAME = "Team Name Here"   # ← change this

game = new_game_state(TEAM_NAME)

print(f"✅  Team registered: {TEAM_NAME}")
print(f"    Starting trust score: {game['trust_score']}/100")
print(f"    Scroll to CELL 2 to configure Frame 1.")


## 🏗️ FRAME 1 — Architecture Config

You are the implementation team for FreshCart.
You designed this system in COSE 260. Now you're building it.

Fill out the config object below. Each field lists options as comments.
**Choose one value per field. Run the cell when done.**


In [ ]:
system_profile = {

    "environment": "consumer",
    # consumer    — fast market, high volume, volatile trust
    # enterprise  — contract-driven, stable, reputational risk
    # government  — fixed budget, procurement rules, political exposure

    "team_structure": "stream_aligned",
    # stream_aligned  — small teams, end-to-end ownership, low coordination cost
    # platform        — shared services team supporting others
    # siloed          — functional departments, high handoff cost

    "build_buy_configure": "build",
    # build       — we write it ourselves
    # buy         — we purchase a product
    # configure   — we implement a vendor platform

    "primary_risk": "delivery_speed",
    # delivery_speed   — we might be too slow
    # technical_debt   — we might cut corners
    # integration      — our components might not fit together
    # vendor_lock      — we might become too dependent on third parties
    # team_capability  — we might not have the right skills

    "data_architecture": "shared_db",
    # dedicated_dbs  — each service owns its data — higher cost, clean boundaries
    # shared_db      — single shared database — lower cost, faster to build

    "coupling": "medium",   # fixed — do not change
}

is_valid, errors, warnings = validate_frame1(system_profile)

if errors:
    print("❌  Fix these errors:\n")
    for e in errors: print(f"    {e}")
else:
    game["frame1"] = system_profile
    game["environment"] = system_profile["environment"]
    print("✅  Frame 1 accepted.\n")
    if warnings:
        for w in warnings: print(f"  {w}")
    print(f"\n    Environment:    {system_profile['environment']}")
    print(f"    Team:           {system_profile['team_structure']}")
    print(f"    Strategy:       {system_profile['build_buy_configure']}")
    print(f"    Risk:           {system_profile['primary_risk']}")
    print(f"    Data:           {system_profile['data_architecture']}")
    print(f"\n    Phase 1 baseline: ${_baseline(system_profile['environment']):,}")
    save_game(game)


## ⚠️ PHASE 1 — Build

FreshCart is under construction. Your architecture choices are now meeting reality.


In [ ]:
# CELL 3 — Phase 1 card
_phase1_cards = select_build_cards(system_profile, game)
_current_card_id = _phase1_cards[0] if _phase1_cards else None

if _current_card_id:
    _card = get_card(_current_card_id)
    game["cards_fired"].append(_current_card_id)
    print_card(_card["name"], _card["scenario"], _card["options"], _card["flavor"])
    print(f"\n    Scroll to CELL 4 to enter your decision.")
else:
    print("✅  No incidents this phase. Scroll to CELL 5.")


In [ ]:
# CELL 4 — Phase 1 decision

decision_rationale = "Enter one sentence explaining why you chose this option"
decision = "a"   # ← change to a, b, or c

if decision_rationale.startswith("Enter"):
    print("❌  decision_rationale required.")
elif decision not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
elif _current_card_id:
    _card = get_card(_current_card_id)
    game["decisions"][_current_card_id] = decision
    game["rationales"][_current_card_id] = decision_rationale
    _env = game["environment"]
    _idx = ENV_IDX[_env]
    _loss_triplet = _card["income_loss"].get(decision, (0,0,0))
    _income_loss = int(_loss_triplet[_idx] * _card.get("environment_modifier",{}).get(_env,1.0))
    _trust_delta = _card["trust_delta"].get(decision, 0)
    _seed = _card.get("seeds",{}).get(decision)
    if _seed: plant_seed(game, _seed)
    update_income(game, "phase1", _baseline(_env) - _income_loss)
    update_trust(game, _trust_delta)
    add_trace(game, _current_card_id,
              f"Phase 1: {_current_card_id} — {system_profile.get('team_structure','')}",
              severity="minor" if _income_loss < 100_000 else "major",
              next_risk=_seed)
    game["facilitator_trace"][-1]["phase"] = "Phase 1"
    print_consequence(_card["name"], decision, _income_loss, _trust_delta,
                      consequence_note=_card.get("consequence_notes",{}).get(decision,""),
                      seed_planted=_seed)
    print_income_summary(game)
    save_game(game)
    print("\n    Scroll to CELL 5 when instructed.")
else:
    print("✅  No card active.")


## 🔧 FRAME 2 — Pipeline Config

You have seen Phase 1. Now configure your pipeline.

**Engineering Capacity hard cap: 100 units.**
See the cost table in the cell below before choosing.


In [ ]:
# CELL 5 — Frame 2 config

# ┌─────────────────────────────────────────┬──────────┬──────────┐
# │  Choice                                 │  Cap cost│  Speed   │
# ├─────────────────────────────────────────┼──────────┼──────────┤
# │  testing: minimal                       │     5    │  fast    │
# │  testing: standard                      │    15    │  moderate│
# │  testing: thorough                      │    30    │  slow    │
# │  deploy:  big_bang                      │     5    │  fastest │
# │  deploy:  rolling                       │    15    │  moderate│
# │  deploy:  blue_green                    │    25    │  moderate│
# │  deploy:  canary                        │    30    │  slowest │
# │  rollback: none                         │     0    │  —       │
# │  rollback: partial                      │    10    │  —       │
# │  rollback: full                         │    20    │  —       │
# │  observability: none                    │     0    │  —       │
# │  observability: basic                   │    10    │  —       │
# │  observability: full                    │    25    │  —       │
# │  change_owner: single_person            │     0    │  —       │
# │  change_owner: shared_pair              │    10    │  —       │
# │  change_owner: team_owned               │    20    │  —       │
# └─────────────────────────────────────────┴──────────┴──────────┘

pipeline_profile = {
    "testing_coverage":   "standard",   # minimal | standard | thorough
    "deployment_method":  "rolling",    # big_bang | rolling | blue_green | canary
    "rollback_plan":      "partial",    # none | partial | full
    "observability_level":"basic",      # none | basic | full
    "change_owner":       "shared_pair",# single_person | shared_pair | team_owned
}

is_valid, errors, warnings, capacity_used = validate_frame2(pipeline_profile)
print(f"Engineering Capacity used: {capacity_used}/100 units\n")

if errors:
    print("❌  Fix these errors:\n")
    for e in errors: print(f"    {e}")
else:
    game["frame2"] = pipeline_profile
    print("✅  Frame 2 accepted.\n")
    if warnings:
        for w in warnings: print(f"  {w}")
    print(f"\n    Testing:       {pipeline_profile['testing_coverage']}")
    print(f"    Deployment:    {pipeline_profile['deployment_method']}")
    print(f"    Rollback:      {pipeline_profile['rollback_plan']}")
    print(f"    Observability: {pipeline_profile['observability_level']}")
    print(f"    Change owner:  {pipeline_profile['change_owner']}")
    save_game(game)
    print("\n    Scroll to CELL 6 to begin Phase 2.")


## ⚠️ PHASE 2 — Live

FreshCart is live. Real load. Real users. Real consequences.

⏱️ **One card in this phase runs under a 60-second timer.**


In [ ]:
# CELL 6 — Phase 2 card
_phase2_cards = select_live_cards(system_profile, pipeline_profile, game)
_current_card_p2 = _phase2_cards[0] if _phase2_cards else None

if _current_card_p2 in ["L1_positive","L3_positive"]:
    _card_p2 = get_card(_current_card_p2)
    _env = game["environment"]
    _protected_loss = _card_p2["income_loss"][ENV_IDX[_env]]
    update_income(game, "phase2", _baseline(_env) - _protected_loss)
    print_positive(_card_p2["name"], _card_p2["message"], _protected_loss,
                   "unchanged — your investment paid off")
    print_income_summary(game)
    save_game(game)
    print("\n    Scroll to CELL 8 for Frame 3.")
elif _current_card_p2:
    _card_p2 = get_card(_current_card_p2)
    game["cards_fired"].append(_current_card_p2)
    print_card(_card_p2["name"], _card_p2["scenario"], _card_p2["options"], _card_p2["flavor"])
    if _card_p2.get("timed"):
        print(f"\n    ⏱  TIMED CARD — {_card_p2['timer_seconds']} seconds.")
        print(f"    No entry = option (b) by default.")
    print("\n    Scroll to CELL 7 to enter your decision.")
else:
    print("✅  No incidents this phase.")
    update_income(game, "phase2", _baseline(game["environment"]))
    print_income_summary(game)
    save_game(game)


In [ ]:
# CELL 7 — Phase 2 decision

decision_rationale_p2 = "Enter one sentence explaining why you chose this option"
decision_p2 = "a"   # ← change to a, b, or c

if not _current_card_p2 or _current_card_p2 in ["L1_positive","L3_positive"]:
    print("✅  No decision required.")
elif decision_rationale_p2.startswith("Enter"):
    print("❌  decision_rationale_p2 required.")
elif decision_p2 not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
else:
    _card_p2 = get_card(_current_card_p2)
    game["decisions"][_current_card_p2] = decision_p2
    game["rationales"][_current_card_p2] = decision_rationale_p2
    _env = game["environment"]
    _idx = ENV_IDX[_env]
    if _current_card_p2 == "L1" and decision_p2 == "b":
        import random
        _ok = random.random() > 0.5
        _key = "b_success" if _ok else "b_fail"
        print("🎲  Hotfix:", "WORKED" if _ok else "FAILED — made it worse")
        _loss_triplet = _card_p2["income_loss"].get(_key,(0,0,0))
        _trust_delta  = _card_p2["trust_delta"].get(_key,0)
    else:
        _loss_triplet = _card_p2["income_loss"].get(decision_p2,(0,0,0))
        _trust_delta  = _card_p2["trust_delta"].get(decision_p2,0)
    _income_loss = _loss_triplet[_idx]
    if _current_card_p2 == "L3":
        _income_loss += _card_p2["income_loss_base"]["locked"][_idx]
    _income_loss = int(_income_loss * _card_p2.get("environment_modifier",{}).get(_env,1.0))
    _seed = _card_p2.get("seeds",{}).get(decision_p2)
    if _seed: plant_seed(game, _seed)
    update_income(game, "phase2", _baseline(_env) - _income_loss)
    update_trust(game, _trust_delta)
    add_trace(game, _current_card_p2, f"Phase 2: {_current_card_p2}",
              severity="minor" if _income_loss < 150_000 else "major", next_risk=_seed)
    game["facilitator_trace"][-1]["phase"] = "Phase 2"
    print_consequence(_card_p2["name"], decision_p2, _income_loss, _trust_delta,
                      consequence_note=_card_p2.get("consequence_notes",{}).get(decision_p2,""),
                      seed_planted=_seed)
    print_income_summary(game)
    save_game(game)
    print("\n    Scroll to CELL 8 when instructed.")


## 🛠️ FRAME 3 — Operations Config

You have seen Phases 1 and 2. Configure your operational layer.


In [ ]:
# CELL 8 — Frame 3 config

operations_profile = {
    "vendor_dependency":  "medium",        # low | medium | high
    "fallback_strategy":  "manual",        # none | manual | automated
    "on_call_coverage":   "business_hours",# none | business_hours | follow_the_sun | full_24x7
    "incident_response":  "runbook",       # ad_hoc | runbook | practiced
}

is_valid, errors, warnings = validate_frame3(operations_profile)

if errors:
    print("❌  Fix these errors:\n")
    for e in errors: print(f"    {e}")
else:
    game["frame3"] = operations_profile
    print("✅  Frame 3 accepted.\n")
    if warnings:
        for w in warnings: print(f"  {w}")
    print(f"\n    Vendor dependency:  {operations_profile['vendor_dependency']}")
    print(f"    Fallback strategy:  {operations_profile['fallback_strategy']}")
    print(f"    On-call coverage:   {operations_profile['on_call_coverage']}")
    print(f"    Incident response:  {operations_profile['incident_response']}")
    save_game(game)
    print("\n    Scroll to CELL 9 to begin Phase 3.")


## ⚠️ PHASE 3 — v2 Release

**Universal requirement — same for every team:**

> FreshCart is adding a real-time delivery tracking feature.
> This requires a new Tracking Service, an updated mobile API contract,
> and a change to the Notification Service.
>
> All teams implement this. What happens next depends on what you built.


In [ ]:
# CELL 9 — Phase 3 card
_phase3_cards = select_v2_cards(system_profile, pipeline_profile, game)
_current_card_p3 = _phase3_cards[0] if _phase3_cards else None
_seed_count = count_seeds(game)

if _seed_count >= 2:
    print(f"⚠  Entering Phase 3 with {_seed_count} active seeds. Severity escalated.\n")

if _current_card_p3:
    _card_p3 = get_card(_current_card_p3)
    game["cards_fired"].append(_current_card_p3)
    print_card(_card_p3["name"], _card_p3["scenario"], _card_p3["options"], _card_p3["flavor"])
    print("\n    Scroll to CELL 10 to enter your decision.")
else:
    print("✅  No major incidents this phase. Your earlier decisions held.")
    update_income(game, "phase3", _baseline(game["environment"]))
    print_income_summary(game)
    save_game(game)


In [ ]:
# CELL 10 — Phase 3 decision

decision_rationale_p3 = "Enter one sentence explaining why you chose this option"
decision_p3 = "a"   # ← change to a, b, or c

if not _current_card_p3:
    print("✅  No decision required.")
elif decision_rationale_p3.startswith("Enter"):
    print("❌  decision_rationale_p3 required.")
elif decision_p3 not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
else:
    _card_p3 = get_card(_current_card_p3)
    game["decisions"][_current_card_p3] = decision_p3
    game["rationales"][_current_card_p3] = decision_rationale_p3
    _env = game["environment"]
    _idx = ENV_IDX[_env]
    _loss_triplet = _card_p3["income_loss"].get(decision_p3,(0,0,0))
    _income_loss = _loss_triplet[_idx]
    if _seed_count >= 2:
        _income_loss = int(_income_loss * 1.35)
    _trust_delta = _card_p3["trust_delta"].get(decision_p3,0)
    _recovery    = _card_p3.get("trust_recovery_modifier",{}).get(decision_p3,0)
    _trust_delta += _recovery
    _seed = _card_p3.get("seeds",{}).get(decision_p3)
    if _seed: plant_seed(game, _seed)
    update_income(game, "phase3", _baseline(_env) - _income_loss)
    update_trust(game, _trust_delta)
    add_trace(game, _current_card_p3,
              f"Phase 3: {_current_card_p3} — seeds active: {game['seeds']}",
              seed_trigger=", ".join(game["seeds"]) if game["seeds"] else None,
              severity="critical" if _income_loss > 300_000 else "major",
              next_risk=_seed)
    game["facilitator_trace"][-1]["phase"] = "Phase 3"
    print_consequence(_card_p3["name"], decision_p3, _income_loss, _trust_delta,
                      consequence_note=_card_p3.get("consequence_notes",{}).get(decision_p3,""),
                      seed_planted=_seed)
    print_income_summary(game)
    save_game(game)
    print("\n    Scroll to CELL 11 when instructed.")


## 🚨 PHASE 4 — Scale Event

**Wait for your instructor to read the scenario aloud.**

No new config. Your architecture holds — or it doesn't.


In [ ]:
# CELL 11 — S1: The Moment of Truth
# Run after instructor reads scenario aloud.

def _calculate_s1_outcome(game):
    env   = game["environment"]
    idx   = ENV_IDX[env]
    base  = PHASE_BASELINE[idx]
    seeds = game.get("seeds",[])
    f2    = game.get("frame2",{})
    f3    = game.get("frame3",{})
    score = 0
    deploy = f2.get("deployment_method","big_bang")
    if deploy in ["canary","blue_green"]: score += 2
    elif deploy == "rolling":             score += 1
    rollback = f2.get("rollback_plan","none")
    if rollback == "full":    score += 2
    elif rollback == "partial": score += 1
    obs = f2.get("observability_level","none")
    if obs == "full":   score += 2
    elif obs == "basic": score += 1
    fallback = f3.get("fallback_strategy","none") if f3 else "none"
    if fallback == "automated": score += 2
    elif fallback == "manual":  score += 1
    ir = f3.get("incident_response","ad_hoc") if f3 else "ad_hoc"
    if ir == "practiced": score += 2
    elif ir == "runbook":  score += 1
    effective = score - len(seeds) * 0.5
    is_gov = env == "government"
    if effective >= 7:
        tier, income, td = "thriving", base + int(base*0.68), 10
        narrative = (f"Traffic spiked 35x. Systems flagged it in 90 seconds.\n"
                     f"Automated fallback activated. FreshCart degraded gracefully.\n\n"
                     f"Revenue captured: +${int(base*0.68):,} above baseline.\n"
                     f"Trust: +10.\n\nThis is what you paid for.")
    elif effective >= 4:
        tier, income, td = "surviving", int(base*0.95), 0
        narrative = "Traffic spiked 35x. FreshCart bent but didn't break. Most revenue captured."
    elif effective >= 1:
        tier, income, td = "struggling", int(base*0.6), -15
        narrative = "Traffic spiked 35x. FreshCart struggled. Major revenue loss. Trust eroded."
    else:
        tier, income, td = "collapsing", int(base*0.2), -30
        narrative = ("Traffic spiked 35x. FreshCart collapsed under load.\n"
                     "The accumulated decisions of the past three phases produced this outcome.")
    if is_gov: income = max(income, int(base*0.4))
    seed_notes, extra_loss, extra_trust = [], 0, 0
    if "P1_silent_fraud"  in seeds: extra_loss += (400_000,300_000,240_000)[idx]; extra_trust -= 40; seed_notes.append("P1 silent fraud surfaces publicly.")
    if "V2_mock_data"     in seeds: extra_loss += (200_000,150_000,120_000)[idx]; extra_trust -= 35; seed_notes.append("V2 mock data discovered.")
    if "D1_db_scaling"    in seeds: extra_loss += (150_000,112_000, 90_000)[idx]; extra_trust -= 10; seed_notes.append("Shared DB saturation triggered.")
    if "P3_workaround_live" in seeds: extra_loss += (120_000,90_000,72_000)[idx]; extra_trust -= 10; seed_notes.append("P3 config cascade triggered.")
    income = max(0, income - extra_loss)
    td    += extra_trust
    if is_gov: income = max(income, int(base*0.4))
    return tier, income, td, narrative, seed_notes

_tier, _s1_income, _s1_trust, _s1_narrative, _seed_resolutions = _calculate_s1_outcome(game)
game["cards_fired"].append("S1")

print(f"\n{'━'*50}")
print("PHASE 4 — THE MOMENT OF TRUTH")
print('━'*50)
print(_s1_narrative)
if _seed_resolutions:
    print("\nSeeds resolving now:")
    for s in _seed_resolutions: print(f"  💥 {s}")

update_income(game, "phase4", _s1_income)
update_trust(game, _s1_trust)
print(f"\nOutcome tier: {_tier.upper()}")
print_income_summary(game)
save_game(game)
print("\n    Scroll to CELL 12.")


In [ ]:
# CELL 12 — S2: Trust revealed first
# Room sits with trust score for 30 seconds before revenue.

print(f"\n{'━'*50}")
print("THE BILL ARRIVES")
print('━'*50)
print(f"\nYour Trust Score: {game['trust_score']}/100")
env_label = {"consumer":"Customer Trust","enterprise":"Stakeholder Confidence","government":"Political Capital"}.get(game["environment"],"Trust")
print(f"({env_label})")
print("\n... room discussion ...\n")
print("Run CELL 13 to reveal revenue impact and enter your decision.")


In [ ]:
# CELL 13 — S2 card display
_card_s2 = get_card("S2")
game["cards_fired"].append("S2")
print_card(_card_s2["name"], _card_s2["scenario"], _card_s2["options"], _card_s2["flavor"])
print("\n    Scroll to CELL 14 to enter your decision.")


In [ ]:
# CELL 14 — S2 decision

decision_rationale_s2 = "Enter one sentence explaining why you chose this option"
decision_s2 = "a"   # ← change to a, b, or c

if decision_rationale_s2.startswith("Enter"):
    print("❌  decision_rationale_s2 required.")
elif decision_s2 not in ["a","b","c"]:
    print("❌  decision must be a, b, or c.")
else:
    game["decisions"]["S2"] = decision_s2
    game["rationales"]["S2"] = decision_rationale_s2
    _env = game["environment"]
    _idx = ENV_IDX[_env]
    _loss_triplet = _card_s2["income_loss"].get(decision_s2,(0,0,0))
    _income_loss  = _loss_triplet[_idx]
    _trust_delta  = _card_s2["trust_delta"].get(decision_s2,0)
    _recovery     = _card_s2.get("trust_recovery_modifier",{}).get(decision_s2,0)
    _seed = _card_s2.get("seeds",{}).get(decision_s2)
    if _seed: plant_seed(game, _seed)
    game["income"]["phase4"] = max(0, game["income"]["phase4"] - _income_loss)
    update_trust(game, _trust_delta)
    add_trace(game,"S2","Phase 4: S2 — board accountability",severity="major")
    game["facilitator_trace"][-1]["phase"] = "Phase 4"
    print_consequence(_card_s2["name"], decision_s2, _income_loss, _trust_delta,
                      consequence_note=_card_s2.get("consequence_notes",{}).get(decision_s2,""),
                      seed_planted=_seed)
    if _recovery: print(f"    Trust recovery modifier: +{_recovery} applied.")
    save_game(game)
    print("\n    Scroll to CELL 15 for your final summary.")


## 📊 CELL 15 — Final Summary
*Copy the output below and give it to your instructor.*


In [ ]:
print_final_summary(game)


## 🔁 CELL 16 — Reflection

**Fill this out before starting the second iteration.**


In [ ]:
strategy_change = {
    "what_failed":    "Describe the main thing that went wrong",
    "what_we_change": "Describe the specific config change you are making",
    "why":            "One sentence: why do you believe this change fixes the problem",
}

print("Reflection recorded:")
for k, v in strategy_change.items():
    print(f"  {k}: {v}")
print("\nRun CELL 17 to begin the second iteration.")


## 🔄 CELL 17 — Second Iteration Setup


In [ ]:
if any(v.startswith("Describe") for v in strategy_change.values()):
    print("❌  Complete the reflection in CELL 16 first.")
else:
    game["income"]           = {"phase1":0,"phase2":0,"phase3":0,"phase4":0}
    game["trust_score"]      = 100
    game["cards_fired"]      = []
    game["decisions"]        = {}
    game["rationales"]       = {}
    game["facilitator_trace"]= []
    game["seeds"]            = []
    game["phase"]            = 0
    print("✅  Second iteration ready.")
    print("    Scroll back to CELL 2, update your config, then run all cells again.")
    print("\n    At the end the engine will ask: did your changes fix the problem?")
